# Featune × Predicting Smartphone Addiction

**A complete, reproducible Kaggle Playground workflow.** This notebook uses [Featune on GitHub](https://github.com/zerolovesea/Featune) ([PyPI](https://pypi.org/project/featune/)) to test controlled feature proposals against a baseline, inspect trial evidence, train on all labeled rows, and write `submission.csv`.

Competition: [Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8). The competition metric is ROC AUC. This is an educational baseline, not a claim of leaderboard superiority. Featune's MIT license applies to its code; the default TabPFN weights have a separate license. This notebook uses LightGBM because this dataset exceeds TabPFN v2's row limit.

**Run order:** install → load → describe schema → search → inspect → holdout check → full refit → submission. No LLM key or private service is required. The competition data stays in Kaggle's competition input; this notebook does not republish it.

## 1. Environment

Kaggle has internet enabled for the first install. The same notebook also runs locally from the repository root after `kaggle competitions download ...` and extraction to `runs/kaggle/<episode>/`. `featune==1.0.0` fixes the public API used here. LightGBM is the explicit downstream estimator; Featune itself handles candidate generation, fold-local compilation, validation, trial history, and pipeline export.

In [ ]:
import importlib.util
import subprocess
import sys

for package, requirement in [("featune", "featune==1.0.0"), ("lightgbm", "lightgbm>=4,<5")]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", requirement])

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import featune

print("Featune", featune.__version__)

## 2. Load data and reserve a holdout

`id` identifies rows; it is not a predictive feature. `X` contains only fields available in the competition test set. We stratify a 10% holdout **before** Featune searches. Its labels never enter `study.optimize`; the holdout is a final sanity check, not a feature-selection signal.

In [ ]:
slug = "playground-series-s6e8"
local_dir = Path("runs/kaggle/s6e8")
kaggle_root = Path("/kaggle/input")
mounted = list(kaggle_root.rglob("train.csv")) if kaggle_root.exists() else []
matching = [p.parent for p in mounted if slug in str(p.parent)]
data_dir = matching[0] if matching else local_dir
if not (data_dir / "train.csv").exists():
    raise FileNotFoundError(f"Could not find {slug} CSVs; mounted train files: {mounted}")
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample = pd.read_csv(data_dir / "sample_submission.csv")
target = "addicted_label"
features = list(test.columns.drop("id"))
assert set(train.columns) == set(test.columns) | {target}
assert sample.columns.tolist() == ["id", target]
assert sample["id"].equals(test["id"])
X = train[features]
X_test = test[features]
y = train[target].astype(int)
assert not y.isna().any() and set(y.unique()) == {0, 1}
search_x, holdout_x, search_y, holdout_y = train_test_split(X, y, test_size=0.10, stratify=y, random_state=42)
print(f"{len(train):,} labeled rows | {len(test):,} test rows | {len(features)} features")
print(f"search: {len(search_x):,} | holdout: {len(holdout_x):,} | positive rate: {y.mean():.3f}")
display(X.head(3))

## 3. Describe the feature contract

`DatasetSchema` is a declaration of usable columns and the prediction goal. Each `FieldSchema` has a name, type, and human-readable meaning. These descriptions help an optional LLM sampler, but the random sampler below needs no network call. The target and `id` are deliberately absent, preventing them from entering the compiled pipeline. Descriptions are explanatory; they do not establish causal validity or prove the fields were available at prediction time.

In [ ]:
descriptions = {
    "age": "Age in years",
    "daily_screen_time_hours": "Daily screen time in hours",
    "social_media_hours": "Daily social media use in hours",
    "gaming_hours": "Daily gaming time in hours",
    "work_study_hours": "Daily work or study screen time in hours",
    "sleep_hours": "Sleep duration in hours",
    "notifications_per_day": "Notifications received per day",
    "app_opens_per_day": "App opens per day",
    "weekend_screen_time": "Weekend screen time in hours",
    "gender": "Reported gender category",
    "stress_level": "Self-reported stress category",
    "academic_work_impact": "Whether device use affects academic work",
}
schema = featune.DatasetSchema(
    fields=[
        featune.FieldSchema(
            name=name,
            dtype="numeric" if pd.api.types.is_numeric_dtype(X[name]) else "categorical",
            description=descriptions[name],
        )
        for name in features
    ],
    objective="Predict the supplied smartphone-addiction label from contemporaneous survey and device-use fields.",
    target_definition="Positive class (1) is addicted_label=1; negative class (0) is addicted_label=0.",
    prediction_point="At prediction time, using only the competition test fields.",
)
assert [field.name for field in schema.fields] == features
print(f"Schema includes {len(schema.fields)} permitted fields")

## 4. Search with the same folds for baseline and candidates

- `RandomSampler(seed=42, features_per_trial=1)` proposes one controlled feature per trial. It never executes generated Python.
- `CVEvaluator` uses two stratified folds, ROC AUC, and the **explicit** LightGBM estimator. `importance_repeats=0` avoids expensive permutation passes; it also means this run makes no feature-importance claim.
- `study.optimize(..., n_trials=2)` evaluates the raw-field baseline plus two proposals on the search split. The baseline remains eligible to win. `best_value` is inner-CV AUC, not a Kaggle score.

Two trials keep this public example quick. Raise `n_trials` and CV folds for a more serious search, while retaining a separate holdout and measuring runtime.

In [ ]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.06,
    num_leaves=31,
    random_state=42,
    n_jobs=2,
    verbosity=-1,
)
evaluator = featune.CVEvaluator(
    estimator=model,
    metric="auc",
    cv=2,
    random_state=42,
    importance_repeats=0,
    n_jobs=1,
)
study = featune.create_study(
    metric="auc",
    sampler=featune.RandomSampler(seed=42, features_per_trial=1),
    search_strategy="independent",
    random_state=42,
)
study.optimize(search_x, search_y, schema=schema, evaluator=evaluator, n_trials=2)
print(f"Baseline CV AUC: {study.baseline_trial.value:.6f}")
print(f"Best CV AUC:     {study.best_value:.6f}")

## 5. Inspect what was tested

`trials_dataframe()` includes baseline, completed proposals, failures, values, and resource use. `best_features` is the selected controlled feature set. If no proposal beats the baseline, an empty set is the correct result; do not claim a gain from an unselected feature.

In [ ]:
trials = study.trials_dataframe()
display(trials[[c for c in ["number", "stage", "state", "value", "delta", "error"] if c in trials]])
print("Selected controlled features:", study.best_features)
print("Completed candidate trials:", sum(t.state == "COMPLETE" and t.number > 0 for t in study.trials))

## 6. Check the untouched holdout once

`export_pipeline()` returns the **fitted** selected sklearn pipeline from the search training rows. It includes Featune's compiler and downstream model, so `predict_proba` applies the same transformations to unseen rows. This holdout AUC is a local diagnostic; it is not the public or private leaderboard score.

In [ ]:
search_pipeline = study.export_pipeline()
holdout_pred = search_pipeline.predict_proba(holdout_x)[:, 1]
holdout_auc = roc_auc_score(holdout_y, holdout_pred)
print(f"Untouched holdout ROC AUC: {holdout_auc:.6f}")

## 7. Refit on **every** labeled row and create the submission

Only after the holdout check, `clone()` copies the selected pipeline structure without learned weights. `.fit(X, y)` trains that structure on the entire official training CSV. The target encoding stays separate from the predictors. We preserve the official sample submission's IDs and header, then validate row count and probability bounds before saving.

In [ ]:
final_pipeline = clone(search_pipeline).fit(X, y)
pred = final_pipeline.predict_proba(X_test)[:, 1]
assert len(pred) == len(test) and np.isfinite(pred).all()
assert ((0 <= pred) & (pred <= 1)).all()
submission = sample.copy()
submission[target] = pred
output = (
    Path("/kaggle/working/submission.csv")
    if Path("/kaggle/working").exists()
    else data_dir / "submission.csv"
)
submission.to_csv(output, index=False)
assert pd.read_csv(output).shape == sample.shape
print(f"Saved {len(submission):,} predictions to {output}")
display(submission.head())

## Reproduce and extend

- Project: [Featune source, documentation, and examples](https://github.com/zerolovesea/Featune) · [install from PyPI](https://pypi.org/project/featune/)
- Competition: [Predicting Smartphone Addiction](https://www.kaggle.com/competitions/s6e8)
- Featune APIs shown: `DatasetSchema`, `FieldSchema`, `RandomSampler`, `CVEvaluator`, `create_study`, `optimize`, `trials_dataframe`, `best_features`, and `export_pipeline`.
- For data with group or time dependencies, replace the random stratified folds with a suitable grouped or time-aware split. For an LLM-guided experiment, see the repository guides and provide credentials privately; never place an API key in a public notebook.

The notebook publishes code and aggregate metrics only. It does not bundle competition CSVs, a fitted model, or credentials.